In [1]:
import numpy as np
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
import re

faq_data = [
    ("How to reset password?", "Click on 'Forgot Password' and follow steps."),
    ("How to change email?", "Go to settings and update your email."),
    ("Payment failed what to do?", "Retry payment or contact support."),
    ("How to login?", "Enter your credentials on login page."),
    ("Account locked solution?", "Wait 30 mins or reset password.")
]

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text.split()

sentences = [preprocess(q) for q, _ in faq_data]

model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)

def sentence_vector(sentence):
    words = preprocess(sentence)
    vectors = []

    for word in words:
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)

faq_vectors = []
for question, answer in faq_data:
    vec = sentence_vector(question)
    faq_vectors.append((vec, answer))

def get_best_answer(user_query):
    query_vec = sentence_vector(user_query)

    best_score = -1
    best_answer = "Sorry, I didn't understand."

    for vec, answer in faq_vectors:
        score = cosine_similarity([query_vec], [vec])[0][0]

        if score > best_score:
            best_score = score
            best_answer = answer

    return best_answer, best_score

while True:
    query = input("\nAsk something (type 'exit' to quit): ")

    if query.lower() == 'exit':
        break

    answer, score = get_best_answer(query)
    print(f"Answer: {answer}")
    print(f"Confidence: {score:.2f}")


Ask something (type 'exit' to quit):  login


Answer: Enter your credentials on login page.
Confidence: 0.60



Ask something (type 'exit' to quit):  password


Answer: Click on 'Forgot Password' and follow steps.
Confidence: 0.50



Ask something (type 'exit' to quit):  exit
